In [3]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.cuda.amp import GradScaler
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
from huggingface_hub import HfApi, upload_file

In [ ]:
import os
from sklearn.model_selection import train_test_split
import random

trainA_dir = '/kaggle/input/datasets/darren2020/ct-to-mri-cgan/Dataset/images/trainA'
trainB_dir = '/kaggle/input/datasets/darren2020/ct-to-mri-cgan/Dataset/images/trainB'
testA_dir  = '/kaggle/input/datasets/darren2020/ct-to-mri-cgan/Dataset/images/testA'
testB_dir  = '/kaggle/input/datasets/darren2020/ct-to-mri-cgan/Dataset/images/testB'

# collect all CT (A) and MRI (B) paths separately
all_A = (
    [os.path.join(trainA_dir, f) for f in os.listdir(trainA_dir)] +
    [os.path.join(testA_dir,  f) for f in os.listdir(testA_dir)]
)
all_B = (
    [os.path.join(trainB_dir, f) for f in os.listdir(trainB_dir)] +
    [os.path.join(testB_dir,  f) for f in os.listdir(testB_dir)]
)

# 80/20 split for each domain independently
domain_A_train, domain_A_val = train_test_split(all_A, test_size=0.2, random_state=42)
domain_B_train, domain_B_val = train_test_split(all_B, test_size=0.2, random_state=42)

# shuffle independently — keeps it unpaired
random.seed(0)
random.shuffle(domain_A_train)
random.shuffle(domain_B_train)
random.shuffle(domain_A_val)
random.shuffle(domain_B_val)

print(f"Domain A (CT)  train: {len(domain_A_train)}")
print(f"Domain B (MRI) train: {len(domain_B_train)}")
print(f"Domain A (CT)  val:   {len(domain_A_val)}")
print(f"Domain B (MRI) val:   {len(domain_B_val)}")

print(f"Total CT  images: {len(all_A)}")
print(f"Total MRI images: {len(all_B)}")

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np

# one sample from each domain
sample_A = Image.open(domain_A_train[0])
sample_B = Image.open(domain_B_train[0])

print("CT  image — size:", sample_A.size, "| mode:", sample_A.mode)
print("MRI image — size:", sample_B.size, "| mode:", sample_B.mode)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(np.array(sample_A), cmap='gray')
axes[0].set_title("Domain A — CT")
axes[0].axis('off')

axes[1].imshow(np.array(sample_B), cmap='gray')
axes[1].set_title("Domain B — MRI")
axes[1].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
class BrainScanDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        img = Image.open(self.image_paths[index]).convert('L')
        img = img.resize((128, 128), Image.BILINEAR)  # resize before anything else
        if self.transform:
            img = self.transform(img)
        return img

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

In [ ]:
dataset_A_train = BrainScanDataset(domain_A_train, transform)
dataset_B_train = BrainScanDataset(domain_B_train, transform)
dataset_A_val   = BrainScanDataset(domain_A_val,   transform)
dataset_B_val   = BrainScanDataset(domain_B_val,   transform)

loader_A_train = DataLoader(dataset_A_train, batch_size=8, shuffle=True, 
                            num_workers=2, pin_memory=True)
loader_B_train = DataLoader(dataset_B_train, batch_size=8, shuffle=True, 
                            num_workers=2, pin_memory=True)
loader_A_val   = DataLoader(dataset_A_val, batch_size=8, 
                            num_workers=2, pin_memory=True)
loader_B_val   = DataLoader(dataset_B_val, batch_size=8, 
                            num_workers=2, pin_memory=True)

In [ ]:
batch_A=next(iter(loader_A_train))
batch_B=next(iter(loader_B_train))

print("Batch A shape:", batch_A.shape)
print("Batch B shape:", batch_B.shape)
print("Batch A min/max:", batch_A.min().item(), batch_A.max().item())
print("Batch B min/max:", batch_B.min().item(), batch_B.max().item())

In [ ]:
class ResNetBlock(nn.Module):
    def __init__(self, channels):
        super(ResNetBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.InstanceNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.InstanceNorm2d(channels)
        )

    def forward(self, x):
        return x+self.block(x)
        

In [ ]:
class Generator(nn.Module):
    def __init__(self, num_residual_blocks=6):
        super(Generator, self).__init__()

        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=7, padding=3),       # 3 → 1
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.bottleneck = nn.Sequential(
            ResNetBlock(256),
            ResNetBlock(256),
            ResNetBlock(256),
            ResNetBlock(256),
            ResNetBlock(256),
            ResNetBlock(256)
        )

        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.InstanceNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=7, padding=3),       # 3 → 1
            nn.Tanh()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.bottleneck(x)
        x = self.decoder(x)
        return x

In [ ]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=4, stride=2, padding=1),   # 3 → 1
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=1, padding=1),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
G_AB = Generator()
G_BA = Generator()
D_A  = Discriminator()
D_B  = Discriminator()

device = 'cuda' if torch.cuda.is_available() else 'cpu'

G_AB = G_AB.to(device)
G_BA = G_BA.to(device)
D_A  = D_A.to(device)
D_B  = D_B.to(device)

# use both GPUs
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    G_AB = nn.DataParallel(G_AB)
    G_BA = nn.DataParallel(G_BA)
    D_A  = nn.DataParallel(D_A)
    D_B  = nn.DataParallel(D_B)

print("Using device:", device)
print("GPU count:", torch.cuda.device_count())

In [ ]:
criterion_GAN=nn.MSELoss()
criterion_cycle=nn.L1Loss()
criterion_identity=nn.L1Loss()

optimizer_G=torch.optim.Adam(
    list(G_AB.parameters())+list(G_BA.parameters()),
    lr=0.0002,
    betas=(0.5,0.999)
)
optimizer_D_A=torch.optim.Adam(
    D_A.parameters(),
    lr=0.0002,
    betas=(0.5,0.999)
)
optimizer_D_B=torch.optim.Adam(
    D_B.parameters(),
    lr=0.0002,
    betas=(0.5,0.999)
)

In [ ]:
from torch.cuda.amp import GradScaler
scaler = GradScaler()
NUM_EPOCHS=50
LAMBDA_CYCLE=10
LAMBDA_IDENTITY=5

In [ ]:
!pip install huggingface_hub -q
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
secret_label = "HF_TOKEN"
secret_value=UserSecretsClient().get_secret(secret_label)

login(token=secret_value)  

REPO_ID = "Awan8754/cycle-gan"  
api = HfApi()

In [ ]:
#all losses
losses_G   = []
losses_D_A = []
losses_D_B = []
losses_cycle = []
for epoch in range(NUM_EPOCHS):
    for i, (real_A, real_B) in enumerate(zip(loader_A_train, loader_B_train)):

        real_A=real_A.to(device)
        real_B=real_B.to(device)

        #training gen
        optimizer_G.zero_grad()
        with torch.cuda.amp.autocast():
            #gen fake img
            fake_B=G_AB(real_A)
            fake_A=G_BA(real_B)
            
            #adversarial loss
            loss_GAN_AB=criterion_GAN(D_B(fake_B), torch.ones_like(D_B(fake_B)))
            loss_GAN_BA=criterion_GAN(D_A(fake_A), torch.ones_like(D_A(fake_A)))
            loss_GAN = loss_GAN_AB+loss_GAN_BA
            #cycle loss
            #real_A->gen_A->fake_B->gen_B->real_A (difference between real A's)
            rec_A =G_BA(fake_B)
            rec_B= G_AB(fake_A)
            loss_cycle=criterion_cycle(rec_A, real_A)+criterion_cycle(rec_B,real_B)

            #identity loss
            #gen_AB(real_B) should be equal to real_B
            idt_B=G_AB(real_B)
            idt_A=G_BA(real_A)
            loss_identity=criterion_identity(idt_B, real_B) + criterion_identity(idt_A, real_A)

            #total generator loss
            loss_G=loss_GAN+LAMBDA_CYCLE*loss_cycle+LAMBDA_IDENTITY*loss_identity

        #backpropagate
        scaler.scale(loss_G).backward()
        scaler.step(optimizer_G)
        scaler.update()

        #train disc A
        optimizer_D_A.zero_grad()
        with torch.cuda.amp.autocast():
            loss_D_A_real=criterion_GAN(D_A(real_A), torch.ones_like(D_A(real_A)))
            loss_D_A_fake=criterion_GAN(D_A(fake_A.detach()), torch.zeros_like(D_A(fake_A.detach())))
            loss_D_A=(loss_D_A_real+loss_D_A_fake)*0.5
        scaler.scale(loss_D_A).backward()
        scaler.step(optimizer_D_A)
        scaler.update()

        #train disc B
        optimizer_D_B.zero_grad()
        with torch.cuda.amp.autocast():
            loss_D_B_real = criterion_GAN(D_B(real_B), torch.ones_like(D_B(real_B)))
            loss_D_B_fake = criterion_GAN(D_B(fake_B.detach()), torch.zeros_like(D_B(fake_B.detach())))
            loss_D_B = (loss_D_B_real + loss_D_B_fake) * 0.5
        scaler.scale(loss_D_B).backward()
        scaler.step(optimizer_D_B)
        scaler.update()
    # after inner loop, before print
    losses_G.append(loss_G.item())
    losses_D_A.append(loss_D_A.item())
    losses_D_B.append(loss_D_B.item())
    losses_cycle.append(loss_cycle.item())

    if (epoch + 1) % 5 == 0:
        # save locally first
        torch.save(G_AB.state_dict(), f'G_AB_epoch{epoch+1}.pth')
        torch.save(G_BA.state_dict(), f'G_BA_epoch{epoch+1}.pth')
        
        # upload to huggingface
        api.upload_file(
            path_or_fileobj=f'G_AB_epoch{epoch+1}.pth',
            path_in_repo=f'checkpoints/G_AB_epoch{epoch+1}.pth',
            repo_id=REPO_ID,
            repo_type="model"
        )
        api.upload_file(
            path_or_fileobj=f'G_BA_epoch{epoch+1}.pth',
            path_in_repo=f'checkpoints/G_BA_epoch{epoch+1}.pth',
            repo_id=REPO_ID,
            repo_type="model"
        )
        print(f"Checkpoint epoch {epoch+1} uploaded to HuggingFace ✅")
    
    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] | Loss_G: {loss_G.item():.4f} | Loss_D_A: {loss_D_A.item():.4f} | Loss_D_B: {loss_D_B.item():.4f} | Loss_Cycle: {loss_cycle.item():.4f}")

In [ ]:
import matplotlib.pyplot as plt

def visualize(G_AB, G_BA, loader_A, loader_B, device, num_examples=4):
    G_AB.eval()
    G_BA.eval()

    with torch.no_grad():
        real_A = next(iter(loader_A)).to(device)
        real_B = next(iter(loader_B)).to(device)

        fake_B = G_AB(real_A)   # CT → MRI
        rec_A  = G_BA(fake_B)   # MRI → CT (reconstructed)
        num_examples = min(num_examples, real_A.size(0))

        def tensor_to_image(tensor, idx):
            img = tensor[idx].cpu()
            img = img.squeeze(0).numpy()   # [1,H,W] → [H,W] for grayscale
            img = (img + 1) / 2
            return img.clip(0, 1)

        fig, axes = plt.subplots(num_examples, 3, figsize=(12, num_examples * 4))
        titles = ["Input (CT)", "Generated (MRI)", "Reconstructed (CT)"]
        images = [real_A, fake_B, rec_A]

        for j in range(num_examples):
            for col in range(3):
                axes[j, col].imshow(tensor_to_image(images[col], j), cmap='gray')
                axes[j, col].set_title(titles[col])
                axes[j, col].axis('off')

        plt.tight_layout()
        plt.show()

    G_AB.train()
    G_BA.train()

visualize(G_AB, G_BA, loader_A_val, loader_B_val, device)

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(losses_G)
plt.title("Generator Loss")
plt.xlabel("Epoch")

plt.subplot(1, 3, 2)
plt.plot(losses_D_A, label='D_A')
plt.plot(losses_D_B, label='D_B')
plt.title("Discriminator Loss")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 3, 3)
plt.plot(losses_cycle)
plt.title("Cycle Consistency Loss")
plt.xlabel("Epoch")

plt.tight_layout()
plt.show()

In [ ]:
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import numpy as np

def evaluate(G_AB, loader_A, loader_B, device):
    G_AB.eval()
    ssim_scores = []
    psnr_scores = []
    
    with torch.no_grad():
        for real_A, real_B in zip(loader_A, loader_B):
            real_A = real_A.to(device)
            fake_B = G_AB(real_A)
            
            for j in range(real_A.size(0)):
                real_img = ((real_A[j].cpu().permute(1,2,0).numpy() + 1) / 2).clip(0,1)
                fake_img = ((fake_B[j].cpu().permute(1,2,0).numpy() + 1) / 2).clip(0,1)
                
                ssim_scores.append(ssim(real_img, fake_img, channel_axis=2, data_range=1.0))
                psnr_scores.append(psnr(real_img, fake_img, data_range=1.0))
    
    print(f"Average SSIM: {np.mean(ssim_scores):.4f}")
    print(f"Average PSNR: {np.mean(psnr_scores):.4f} dB")
    
    G_AB.train()

# call after training
evaluate(G_AB, loader_A_val, loader_B_val, device)